В цьому домашньому завданні ми знову працюємо з даними з нашого змагання ["Bank Customer Churn Prediction (DLU Course)"](https://www.kaggle.com/t/7c080c5d8ec64364a93cf4e8f880b6a0).

Тут ми побудуємо рішення задачі класифікації з використанням kNearestNeighboors, знайдемо оптимальні гіперпараметри для цього методу і зробимо базові ансамблі. Це дасть змогу порівняти перформанс моделі з попередніми вивченими методами.

0. Зчитайте дані `train.csv` та зробіть препроцесинг використовуючи написаний Вами скрипт `process_bank_churn.py` так, аби в результаті отримати дані в розбитті X_train, train_targets, X_val, val_targets для експериментів.

  Якщо Вам не вдалось реалізувати в завданні `2.3. Дерева прийняття рішень` скрипт `process_bank_churn.py` - можна скористатись готовим скриптом з запропонованого рішення того завдання.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

np.set_printoptions(legacy='1.25')

In [3]:
from process_bank_churn import preprocess_data

In [4]:
raw_df = pd.read_csv("drive/MyDrive/Colab Notebooks/data/train.csv")

In [5]:
data = preprocess_data(raw_df, True)

In [6]:
X_train = data['train_inputs']
train_targets = data['train_targets']
X_val = data['val_inputs']
val_targets = data['val_targets']

1. Навчіть на цих даних класифікатор kNN з параметрами за замовченням і виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах. Зробіть заключення про отриману модель: вона хороша/погана, чи є high bias/high variance?

In [7]:
def get_auroc(model, inputs, targets, title):
    preds = model.predict_proba(inputs)[:, 1]
    fpr, tpr, _ = roc_curve(targets, preds)
    roc_auc = auc(fpr, tpr)
    print(f"{title} ROC AUC: {roc_auc:.2f}")

In [8]:
knn = KNeighborsClassifier()
knn.fit(X_train, train_targets)

KNeighborsClassifier()

In [9]:
get_auroc(knn, X_train, train_targets, 'Train')

Train ROC AUC: 0.96


In [10]:
get_auroc(knn, X_val, val_targets, 'Validation')

Validation ROC AUC: 0.85


Overall, the model performs well, with strong AUROC scores on both the training and validation sets.    
But there is difference (0.11) between these values, which indicates overfitting of model

2. Використовуючи `GridSearchCV` знайдіть оптимальне значення параметра `n_neighbors` для класифікатора `kNN`. Псотавте крос валідацію на 5 фолдів.

  Після успішного завершення пошуку оптимального гіперпараметра
    - виведіть найкраще значення параметра
    - збережіть в окрему змінну `knn_best` найкращу модель, знайдену з `GridSearchCV`
    - оцініть якість передбачень  `knn_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пукнтом (2) цього завдання? Чи є вона краще за дерево прийняття рішень з попереднього ДЗ?

In [11]:
params_knn = {'n_neighbors': np.arange(1, 25)}
knn_gs = GridSearchCV(KNeighborsClassifier(), params_knn, cv=5)
knn_gs.fit(X_train, train_targets)

GridSearchCV(cv=5, estimator=KNeighborsClassifier(),
             param_grid={'n_neighbors': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24])})

In [12]:
print(knn_gs.best_params_, knn_gs.best_score_)

{'n_neighbors': 9} 0.8671666666666666


In [13]:
knn_best = knn_gs.best_estimator_

In [14]:
get_auroc(knn_best, X_train, train_targets, 'Train')

Train ROC AUC: 0.94


In [15]:
get_auroc(knn_best, X_val, val_targets, 'Validation')

Validation ROC AUC: 0.88


Model became better in comparison to previous one, but it shows less accuracy to model from previous home task   
(AUROC for Training: 0.93, AUROC for Validation: 0.92)

3. Виконайте пошук оптимальних гіперпараметрів для `DecisionTreeClassifier` з `GridSearchCV` за сіткою параметрів
  - `max_depth` від 1 до 20 з кроком 2
  - `max_leaf_nodes` від 2 до 10 з кроком 1

  Обовʼязково при цьому ініціюйте модель з фіксацією `random_state`.

  Поставте кросвалідацію на 3 фолди, `scoring='roc_auc'`, та виміряйте, скільки часу потребує пошук оптимальних гіперпараметрів.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення параметра
    - збережіть в окрему змінну `dt_best` найкращу модель, знайдену з `GridSearchCV`
    - оцініть якість передбачень  `dt_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи ця модель краща за ту, що ви знайшли вручну?

In [16]:
params_tree = {'max_depth': np.arange(1, 21, 2), 'max_leaf_nodes': np.arange(2, 11)}
tree_gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params_tree, cv=3, scoring='roc_auc', verbose=1)

In [17]:
%%time
tree_gs.fit(X_train, train_targets)

Fitting 3 folds for each of 90 candidates, totalling 270 fits
CPU times: user 5.33 s, sys: 17.1 ms, total: 5.35 s
Wall time: 5.35 s


GridSearchCV(cv=3, estimator=DecisionTreeClassifier(random_state=42),
             param_grid={'max_depth': array([ 1,  3,  5,  7,  9, 11, 13, 15, 17, 19]),
                         'max_leaf_nodes': array([ 2,  3,  4,  5,  6,  7,  8,  9, 10])},
             scoring='roc_auc', verbose=1)

In [18]:
print(tree_gs.best_params_, tree_gs.best_score_)

{'max_depth': 5, 'max_leaf_nodes': 10} 0.9013929183420709


In [19]:
dt_best = tree_gs.best_estimator_

In [20]:
get_auroc(dt_best, X_train, train_targets, 'Train')

Train ROC AUC: 0.90


In [21]:
get_auroc(dt_best, X_val, val_targets, 'Validation')

Validation ROC AUC: 0.90


This model is even better, but still there is space for improvements

4. Виконайте пошук оптимальних гіперпараметрів для `DecisionTreeClassifier` з `RandomizedSearchCV` за заданою сіткою параметрів і кількість ітерацій 40.

  Поставте кросвалідацію на 3 фолди, `scoring='roc_auc'`, зафіксуйте `random_seed` процедури крос валідації та виміряйте, скільки часу потребує пошук оптимальних гіперпараметрів.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення параметра
    - збережіть в окрему змінну `dt_random_search_best` найкращу модель, знайдену з `RandomizedSearchCV`
    - оцініть якість передбачень  `dt_random_search_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи ця модель краща за ту, що ви знайшли з `GridSearch`?
    - проаналізуйте параметри `dt_random_search_best` і порівняйте з параметрами `dt_best` - яку бачите відмінність? Ця вправа потрібна аби зрозуміти, як різні налаштування `DecisionTreeClassifier` впливають на якість моделі.

In [22]:
params_dt = {
    'criterion': ['gini', 'entropy'],
    'splitter': ['best', 'random'],
    'max_depth': np.arange(1, 20),
    'max_leaf_nodes': np.arange(2, 20),
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': [None, 'sqrt', 'log2']
}

In [23]:
tree_rs = RandomizedSearchCV(
    DecisionTreeClassifier(random_state=42),
    params_dt,
    n_iter = 40,
    cv=3,
    scoring="roc_auc",
    random_state=42,
    verbose=1
)

In [24]:
%%time
tree_rs.fit(X_train, train_targets)

Fitting 3 folds for each of 40 candidates, totalling 120 fits
CPU times: user 1.51 s, sys: 11 ms, total: 1.52 s
Wall time: 1.52 s


RandomizedSearchCV(cv=3, estimator=DecisionTreeClassifier(random_state=42),
                   n_iter=40,
                   param_distributions={'criterion': ['gini', 'entropy'],
                                        'max_depth': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19]),
                                        'max_features': [None, 'sqrt', 'log2'],
                                        'max_leaf_nodes': array([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18,
       19]),
                                        'min_samples_leaf': [1, 2, 4, 8],
                                        'min_samples_split': [2, 5, 10, 20],
                                        'splitter': ['best', 'random']},
                   random_state=42, scoring='roc_auc', verbose=1)

In [25]:
tree_rs.best_params_, tree_rs.best_score_

({'splitter': 'best',
  'min_samples_split': 20,
  'min_samples_leaf': 2,
  'max_leaf_nodes': 14,
  'max_features': None,
  'max_depth': 16,
  'criterion': 'entropy'},
 0.910864318350194)

In [26]:
dt_random_search_best = tree_rs.best_estimator_

In [27]:
get_auroc(dt_random_search_best, X_train, train_targets, 'Train')

Train ROC AUC: 0.92


In [28]:
get_auroc(dt_random_search_best, X_val, val_targets, 'Validation')

Validation ROC AUC: 0.92


When comparing the parameters used for `dt_random_search_best` and `dt_best`, we observe that `dt_best` relies on a limited set of hyperparameters. This results in a faster search process with fewer combinations, but may come at the cost of reduced model accuracy. In contrast, the broader parameter space explored in `dt_random_search_best` allows for a more detailed and comprehensive search, increasing the likelihood of finding a better-performing model.

This model showed the best result

5. Якщо у Вас вийшла метрика `AUROC` в цій серії експериментів - зробіть ще один `submission` на Kaggle і додайте код для цього і скріншот скора на публічному лідерборді нижче.

  Сподіваюсь на цьому етапі ви вже відчули себе справжнім дослідником 😉

In [29]:
from process_bank_churn import preprocess_new_data

In [30]:
test_raw_df = pd.read_csv("drive/MyDrive/Colab Notebooks/data/test.csv")

In [31]:
X_test_input = preprocess_new_data(test_raw_df, data['numeric_cols'], data['categorical_cols'], data['encoded_cols'], data['encoder'], data['scaler'])
X_test_input.head(5)

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain,Gender_Female,Gender_Male
0,0.365155,0.214286,0.2,0.696265,0.000000,1.0,1.0,0.789477,0.0,1.0,0.0,0.0,1.0
1,0.286396,0.375000,0.5,0.000000,0.333333,1.0,1.0,0.337131,1.0,0.0,0.0,0.0,1.0
2,0.656325,0.446429,0.8,0.000000,0.333333,1.0,0.0,0.783859,1.0,0.0,0.0,0.0,1.0
3,0.682578,0.482143,0.3,0.000000,0.000000,1.0,1.0,0.834571,0.0,0.0,1.0,0.0,1.0
4,0.384248,0.446429,0.8,0.000000,0.333333,1.0,1.0,0.718421,0.0,0.0,1.0,0.0,1.0


In [32]:
test_prob = dt_random_search_best.predict_proba(X_test_input)[:,1]
test_raw_df['Exited'] = test_prob

In [33]:
submission_raw_df = pd.read_csv("drive/MyDrive/Colab Notebooks/data/sample_submission.csv")
submission_raw_df['Exited'] = test_raw_df['Exited']
submission_raw_df['Exited'].head()

,Exited
0,0.237911
1,0.012115
2,0.203947
3,0.569848
4,0.082171


In [34]:
submission_raw_df.to_csv('submission_cross_val.csv', index=False)